In [1]:
import sys
import os
# Add src directory to path
sys.path.append(os.path.dirname(os.getcwd()))


In [2]:
print(sys.path)

['/home/abhishek/snap/code/205/.local/share/uv/python/cpython-3.11.13-linux-x86_64-gnu/lib/python311.zip', '/home/abhishek/snap/code/205/.local/share/uv/python/cpython-3.11.13-linux-x86_64-gnu/lib/python3.11', '/home/abhishek/snap/code/205/.local/share/uv/python/cpython-3.11.13-linux-x86_64-gnu/lib/python3.11/lib-dynload', '', '/home/abhishek/Downloads/work/active_gliner/.venv/lib/python3.11/site-packages', '/home/abhishek/Downloads/work/active_gliner']


In [3]:
from src.preprocess.data_loading import load_mit_dataset, load_results_from_file

In [4]:
results_path = "../results/final_results_w_analysis.json"
results = load_results_from_file(results_path)


In [5]:
import pandas as pd
results.keys()
results_df=pd.DataFrame(results)
display(results_df.head())

,no_corrected_examples,no_generated_examples,final_f1,final_confidence,cache_used,analysis_integrated
0,10,0,0.291219,0.954087,False,False
1,10,10,0.272561,0.936729,True,True
2,10,25,0.214871,0.940335,True,True
3,10,50,0.242052,0.946416,True,True
4,10,100,0.250925,0.920177,True,True


In [6]:
import logging
logging.basicConfig(level=logging.INFO)
from src.utils.device_setup import setup_device

setup=setup_device()
print(setup)

INFO:ActiveLearning:Using device: cuda
INFO:ActiveLearning:CUDA version: 12.8
INFO:ActiveLearning:Number of GPUs visible: 1
INFO:ActiveLearning:Current GPU: 0
INFO:ActiveLearning:GPU Name: NVIDIA GeForce RTX 3050 6GB Laptop GPU
INFO:ActiveLearning:GPU Memory: 5.7 GB


cuda


In [7]:
from src.utils.reproducibility import set_all_seeds
seeds=set_all_seeds(42)

INFO:ActiveLearning:Setting all seeds to 42 for reproducibility...


In [8]:
from src.utils.logging_setup import setup_logging
logger=setup_logging()

INFO:ActiveLearning:================================================================================
INFO:ActiveLearning:ACTIVE LEARNING PIPELINE STARTED
INFO:ActiveLearning:================================================================================
INFO:ActiveLearning:Log file: logs/active_learning_20250908_142337.log


In [9]:
mit_data_path="../data/mit-movie/"
train_data_path=os.path.join(mit_data_path,"train.json")
val_data_path=os.path.join(mit_data_path,"dev.json")
test_data_path=os.path.join(mit_data_path,"test.json")
labels_path=os.path.join(mit_data_path,"labels.json")


train_data,entity_types=load_mit_dataset(train_data_path,labels_path)
val_data,entity_types=load_mit_dataset(val_data_path,labels_path)
test_data,entity_types=load_mit_dataset(test_data_path,labels_path)
test_data[0]

INFO:ActiveLearning:Loading train data from: ../data/mit-movie/train.json
INFO:ActiveLearning:Processed 9774 examples
INFO:ActiveLearning:Entity types: ['genre', 'year', 'plot', 'average ratings', 'actor', 'title', 'song', 'character', 'rating', 'review', 'director', 'trailer']
INFO:ActiveLearning:Loading train data from: ../data/mit-movie/dev.json
INFO:ActiveLearning:Processed 2442 examples
INFO:ActiveLearning:Entity types: ['genre', 'year', 'plot', 'average ratings', 'actor', 'title', 'song', 'character', 'rating', 'review', 'director', 'trailer']
INFO:ActiveLearning:Loading train data from: ../data/mit-movie/test.json
INFO:ActiveLearning:Processed 2442 examples
INFO:ActiveLearning:Entity types: ['genre', 'year', 'plot', 'average ratings', 'actor', 'title', 'song', 'character', 'rating', 'review', 'director', 'trailer']


{'tokenized_text': ['are',
  'there',
  'any',
  'good',
  'romantic',
  'comedies',
  'out',
  'right',
  'now'],
 'ner': [(4, 5, 'genre'), (7, 8, 'year')]}

In [10]:
from gliner import GLiNER
from gliner.data_processing.collator import DataCollator
from gliner.training import Trainer, TrainingArguments
from transformers import TrainerCallback

model = GLiNER.from_pretrained("knowledgator/modern-gliner-bi-large-v1.0")
model.config.max_len = 8192

if hasattr(model.data_processor, 'transformer_tokenizer'):    
    model.data_processor.transformer_tokenizer.model_max_length = 8192

# Get base parameter count
base_total = sum(p.numel() for p in model.model.parameters())
logger.info(f"Base Parameters: {base_total:,}")



/home/abhishek/Downloads/work/active_gliner/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 75497.47it/s]
INFO:ActiveLearning:Base Parameters: 529,845,248


In [12]:
from src.evaluation.enchanced_eval import enhanced_evaluate


results=enhanced_evaluate(model, test_data, entity_types, batch_size=8,has_ground_truth=True)


Running enhanced evaluation...
Processing 2442 examples...
Analyzing errors with ground truth...


In [13]:
results

{'overall_metrics': {'total_predictions': 3633,
  'overall_confidence': np.float64(0.7867653715915665),
  'overall_confidence_pct': np.float64(78.67653715915665),
  'total_examples': 2442,
  'entity_level_accuracy': np.float64(0.3974823574289529),
  'entity_level_accuracy_pct': np.float64(39.74823574289529),
  'example_level_accuracy': 0.15192465192465193,
  'example_level_accuracy_pct': 15.192465192465193,
  'overall_f1': np.float64(0.46958089229382605),
  'overall_f1_pct': np.float64(46.95808922938261),
  'incorrect_examples': [{'tokenized_text': ['are',
     'there',
     'any',
     'good',
     'romantic',
     'comedies',
     'out',
     'right',
     'now'],
    'ner': [(4, 5, 'genre'), (7, 8, 'year')],
    'predictions': [[4, 5, 'genre']],
    'scores': [0.8240768313407898],
    'errors': {'false_negatives': [[7, 8, 'year']], 'false_positives': []}},
   {'tokenized_text': ['show',
     'me',
     'a',
     'movie',
     'about',
     'cars',
     'that',
     'talk'],
    'ner

In [14]:
f1=results['classification_report_df'][results['classification_report_df']['entity_type'] == 'micro_avg']['f1'].iloc[0]
print(f"F1 Score: {f1:.4f} ({f1*100:.2f}%)")

F1 Score: 0.4696 (46.96%)


In [15]:
from src.evaluation.helper import display_results

display_results(results)


ENHANCED EVALUATION RESULTS

Overall Metrics:
Total Predictions: 3,633
Overall Confidence: 0.7868 (78.68%)
Total Examples: 2,442
Example-Level Accuracy: 0.1519 (15.19%)
Entity-Level Accuracy: 0.3975 (39.75%)
Overall F1 Score: 0.4696 (46.96%)

Confidence Distribution:


,genre,year,plot,average ratings,actor,title,song,character,rating,review,director,trailer
Confidence Range,,,,,,,,,,,,
0-25%,0,0,0,0,0,0,0,0,0,0,0,0
26-50%,0,0,0,0,0,0,0,0,0,0,0,0
51-75%,354,82,219,4,151,15,11,120,171,45,101,18
76-100%,478,196,318,5,468,9,24,163,342,33,186,1



Classification Report:


,entity_type,tp,fp,fn,precision,recall,f1,support,avg_prediction_confidence
0,genre,562,297,555,0.654249,0.503133,0.568826,1117,0.77
1,year,205,83,515,0.711806,0.284722,0.406746,720,0.81
2,plot,138,412,353,0.250909,0.281059,0.265130,491,0.77
3,actor,585,52,227,0.918367,0.720443,0.807453,812,0.82
4,average ratings,1,8,448,0.111111,0.002227,0.004367,449,0.72
5,character,67,230,22,0.225589,0.752809,0.347150,89,0.78
6,title,2,26,559,0.071429,0.003565,0.006791,561,0.69
7,song,20,16,34,0.555556,0.370370,0.444444,54,0.80
8,rating,205,328,203,0.384615,0.502451,0.435707,408,0.79
9,review,6,77,50,0.072289,0.107143,0.086331,56,0.73



True Positives Confidence Analysis:


,genre,year,plot,average ratings,actor,title,song,character,rating,review,director,trailer
Confidence Range,,,,,,,,,,,,
0-25%,0,0,0,0,0,0,0,0,0,0,0,0
26-50%,0,0,0,0,0,0,0,0,0,0,0,0
51-75%,164,22,63,1,134,2,3,10,72,1,98,4
76-100%,388,179,73,0,437,0,17,56,123,4,184,0



False Positives Confidence Analysis:


,genre,year,plot,average ratings,actor,title,song,character,rating,review,director,trailer
Confidence Range,,,,,,,,,,,,
0-25%,0,0,0,0,0,0,0,0,0,0,0,0
26-50%,0,0,0,0,0,0,0,0,0,0,0,0
51-75%,190,60,156,3,17,13,8,110,99,44,3,14
76-100%,90,17,245,5,31,9,7,107,219,29,2,1



Incorrect Examples: 2071
Corrected Labels Available: 2071


In [16]:
results_org=model.evaluate(test_data,
            flat_ner=True,
            threshold=0.5,
            batch_size=8,
            entity_types=entity_types)

In [17]:
results_org

('P: 57.36%\tR: 39.75%\tF1: 46.96%\n', np.float64(0.46958089229382605))

In [18]:
from src.lora.lora_parameters import get_lora_config,apply_lora_to_model

lora_config = get_lora_config()
print(lora_config)

model,base_total,lora_traniable = apply_lora_to_model(model)

INFO:ActiveLearning:Applying LoRA Configuration...
INFO:ActiveLearning:Base Parameters: 529,845,248
INFO:ActiveLearning:Trainable Parameters: 20,815,872 (3.9% of original)


LoraConfig(task_type=<TaskType.TOKEN_CLS: 'TOKEN_CLS'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path=None, revision=None, inference_mode=False, r=32, target_modules={'Wqkv', 'span_rep_layer.span_rep_layer.project_end.3', 'projection', 'span_rep_layer.span_rep_layer.project_end.0', 'prompt_rep_layer.0', 'intermediate.dense', 'Wi', 'key', 'Wo', 'span_rep_layer.span_rep_layer.out_project.3', 'output.dense', 'span_rep_layer.span_rep_layer.out_project.0', 'span_rep_layer.span_rep_layer.project_start.3', 'dense', 'query', 'prompt_rep_layer.3', 'value', 'span_rep_layer.span_rep_layer.project_start.0'}, exclude_modules=None, lora_alpha=64, lora_dropout=0.1, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_confi